# Photo Sorter — Colab (free, private, wipe-after-sort)

**What this is:** the cloud half of Photo Sorter v0.4. Your phone makes
`thumbnails.zip` (max 384 px JPEGs, **not** full photos) + `manifest.csv`
and you upload that zip here. This notebook sorts them with
**NudeNet + a vision model**, writes `plan.csv` / `plan.json`, you
download the plan, and the last cell **wipes the thumbnails out of
this runtime**.

**Privacy — read this first, it is not anonymous:**
- Free Colab = a Google server while the runtime is alive. A 384 px
  thumbnail still clearly shows faces, places, text and NSFW content.
  Uploading thumbnails is uploading the photos, just smaller.
- This notebook **never mounts Google Drive and never saves anything
  to Drive**. Everything lives in `/content` (the runtime's temp disk)
  and the wipe cell deletes it. Colab runtimes are recycled after
  inactivity, but you cannot independently verify Google deleted it —
  the wipe cell only proves `/content` is empty afterwards.
- NSFW rows in the plan have caption `[redacted]` and
  `needs_review=true`. Nothing here is auto-deleted on your phone;
  the plan is a proposal you approve per category in the app.
- If you do not want *any* image on a server, do not use this
  notebook — use the phone's on-device Scan instead (slower, worse
  categories, fully local).

**How to use it (5 steps):**
1. In the Photo Sorter app tap the blue **Upload to notebook** button.
   It zips thumbnails + `manifest.csv` to `Download/PhotoSorter-Colab/`
   and opens this notebook. The upload itself runs as a background
   worker on the phone, so rotating the phone or leaving the app
   does not stop it.
2. Run the cells below in order. In the upload cell, pick
   `thumbnails.zip` from your phone / computer.
3. Wait for the NudeNet + vision run. It checkpoints every 50 photos
   to `plan_partial.csv`, so a disconnect does not lose everything.
4. Download `plan.csv` from the download cell.
5. **Run the wipe cell before closing the tab.**
6. Back in the app, tap the blue **Sort** button, pick the downloaded
   `plan.csv`, and it sorts from the plan in a background worker
   (survives rotation / app kill / reboot — it resumes from the plan).


In [ ]:
# 0 — Check the runtime (free Colab: T4 GPU if you picked one)
# Runtime > Change runtime type > T4 GPU is recommended. CPU works, it is just slow.
import torch, sys
print("python", sys.version)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("No GPU — the vision step will fall back to CLIP on CPU (slower, coarser).")


In [ ]:
# 1 — Install: NudeNet + vision models (runs once per runtime, ~2-4 min)
# NudeNet = NSFW pre-screen. Qwen2.5-VL-3B = the vision classifier.
# If Qwen fails to load later, the notebook falls back to CLIP zero-shot.
!pip install -q nudenet transformers accelerate torch pillow pandas tqdm
print("installed")


In [ ]:
# 2 — Upload thumbnails.zip from the phone app, unzip it here (in /content only)
from google.colab import files
import zipfile, pathlib, csv, hashlib, os

uploaded = files.upload()  # pick thumbnails.zip
assert len(uploaded) == 1, "Upload exactly one file: thumbnails.zip"
zip_path = list(uploaded.keys())[0]
print("uploaded:", zip_path, os.path.getsize(zip_path), "bytes")

thumbs = pathlib.Path("/content/thumbs")
thumbs.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(thumbs)
# manifest.csv maps zip filename -> original photo name / MediaStore id
manifest = {}
mp = thumbs / "manifest.csv"
if mp.exists():
    with open(mp, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            manifest[row["zip_name"]] = row
    print("manifest rows:", len(manifest))
else:
    print("WARNING: no manifest.csv in the zip — plan rows will use the zip filename only.")

images = sorted(p for p in thumbs.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"} and p.name != "manifest.csv")
print("images found:", len(images))
for p in images[:5]:
    print(" ", p.name, p.stat().st_size, "bytes")


In [ ]:
# 3 — NudeNet pre-screen (same thresholds as the PC script sort_photos_local.py)
# score >= 0.5 -> explicit, skip the vision model, caption [redacted]
# score >= 0.2 -> needs_review, vision model adjudicates (statue / art / swimwear etc.)
from nudenet import NudeDetector
import json, pathlib

NSFW_HIGH, NSFW_MID = 0.5, 0.2
detector = NudeDetector()
scores = {}
ckpt = pathlib.Path("/content/nudenet_scores.json")
if ckpt.exists():
    scores = json.loads(ckpt.read_text())
    print("resumed NudeNet scores:", len(scores))

for i, p in enumerate(images):
    if p.name in scores:
        continue
    try:
        res = detector.detect(str(p))
        parts = [r["class"] for r in res if float(r["score"]) >= NSFW_MID]
        score = max([float(r["score"]) for r in res], default=0.0)
        scores[p.name] = {"score": round(score, 3), "parts": ";".join(parts)}
    except Exception as e:
        scores[p.name] = {"score": 0.0, "parts": "", "error": str(e)}
    if (i + 1) % 50 == 0:
        ckpt.write_text(json.dumps(scores))
        print(f"  NudeNet {i+1}/{len(images)}")
ckpt.write_text(json.dumps(scores))
print("NudeNet done. explicit (>=0.5):", sum(1 for v in scores.values() if v["score"] >= NSFW_HIGH),
      "| review (>=0.2):", sum(1 for v in scores.values() if v["score"] >= NSFW_MID))


In [ ]:
# 4 — Vision classification: Qwen2.5-VL-3B first, CLIP zero-shot fallback
# One prompt per image, JSON-only reply. Checkpoints every 50 to plan_partial.csv.
# This is the slow step: ~1-3 s/image on a T4, much slower on CPU. Leave the tab open;
# Colab keeps running while the tab is open, and the checkpoint lets you resume.
import json, re, pathlib, pandas as pd
from PIL import Image

CATEGORIES = [
  "nsfw_suggestive/futanari",
  "nsfw_suggestive/other",
  "nsfw_explicit/futanari",
  "nsfw_explicit/other",
  "meme",
  "screenshot",
  "statistics_chart_document",
  "people_family_friends",
  "selfie_vtuber",
  "travel_hong_kong",
  "food",
  "school_coursework",
  "creative_3d_render",
  "game_fandom",
  "chomp_city_robotics",
  "davispedia",
  "pet_animal",
  "landscape_nature",
  "duplicate_near_duplicate",
  "blurry_low_quality",
  "other"
]
SUBCATS = {
  "meme": [
    "anime",
    "reaction",
    "text_meme",
    "other"
  ],
  "screenshot": [
    "app_screenshot",
    "chat_screenshot",
    "game_screenshot",
    "other"
  ],
  "people_family_friends": [
    "family",
    "friends",
    "group",
    "portrait",
    "other"
  ],
  "selfie_vtuber": [
    "selfie",
    "vtuber",
    "cosplay",
    "other"
  ],
  "travel_hong_kong": [
    "hong_kong",
    "travel_other",
    "landmark",
    "other"
  ],
  "food": [
    "meal",
    "drink",
    "snack_dessert",
    "recipe_menu",
    "other"
  ],
  "game_fandom": [
    "one_piece",
    "pokemon",
    "genshin",
    "honkai_star_rail",
    "other_fandom"
  ],
  "creative_3d_render": [
    "blender_render",
    "3d_model",
    "digital_art",
    "other"
  ],
  "school_coursework": [
    "notes",
    "assignment",
    "lecture_slide",
    "other"
  ],
  "statistics_chart_document": [
    "chart",
    "document",
    "table",
    "other"
  ],
  "pet_animal": [
    "cat",
    "dog",
    "other_animal"
  ],
  "landscape_nature": [
    "city",
    "nature",
    "sea_sky",
    "other"
  ],
  "chomp_city_robotics": [
    "robot",
    "wiring",
    "workshop",
    "other"
  ],
  "davispedia": [
    "davispedia",
    "other"
  ],
  "nsfw_suggestive/futanari": [
    "futanari"
  ],
  "nsfw_suggestive/other": [
    "other"
  ],
  "nsfw_explicit/futanari": [
    "futanari"
  ],
  "nsfw_explicit/other": [
    "other"
  ]
}

PROMPT = ("Classify this image into exactly ONE category from this list: "
          + ", ".join(CATEGORIES) +
          ". Then pick a subcategory from the matching list if one applies, else 'other'. "
          "Reply with JSON only, no other text: "
          '{"category": "<one of the list>", "subcategory": "<word>", "confidence": 0.0-1.0, "caption": "<short neutral caption, max 12 words>"}')

def clean(cat, sub, conf, cap):
    cat = cat if cat in CATEGORIES else "other"
    allowed = SUBCATS.get(cat, ["other"])
    sub = sub if sub in allowed else "other"
    return cat, sub, float(conf) if 0 <= float(conf) <= 1 else 0.5, str(cap)[:120]

# --- try Qwen2.5-VL-3B ---
vlm = None
try:
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText
    MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
    print("loading", MODEL, "... (first load downloads ~6 GB)")
    proc = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
    vlm = AutoModelForImageTextToText.from_pretrained(
        MODEL, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto", trust_remote_code=True)
    print("Qwen2.5-VL loaded")
except Exception as e:
    print("Qwen load FAILED, falling back to CLIP zero-shot:", e)
    vlm = None

# --- CLIP fallback (fast, coarser) ---
clip = None
if vlm is None:
    from transformers import CLIPProcessor, CLIPModel
    import torch
    clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    print("CLIP fallback loaded")

def vlm_classify(path):
    img = Image.open(path).convert("RGB")
    if vlm is not None:
        msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]
        text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = proc(text=[text], images=[img], padding=True, return_tensors="pt").to(vlm.device)
        out = vlm.generate(**inputs, max_new_tokens=120, do_sample=False)
        raw = proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        m = re.search(r"\{.*\}", raw, re.S)
        d = json.loads(m.group(0)) if m else {}
        return clean(d.get("category", "other"), d.get("subcategory", "other"),
                     d.get("confidence", 0.5), d.get("caption", ""))
    else:
        import torch
        inputs = clip_proc(text=CATEGORIES, images=img, return_tensors="pt", padding=True)
        with torch.no_grad():
            probs = clip(**inputs).logits_per_image.softmax(dim=1)[0]
        i = int(probs.argmax())
        return CATEGORIES[i], "other", round(float(probs[i]), 2), "clip zero-shot"

rows, done = [], set()
partial = pathlib.Path("/content/plan_partial.csv")
if partial.exists():
    prev = pd.read_csv(partial)
    rows = prev.to_dict("records")
    done = {r["name"] for r in rows}
    print("resumed partial plan:", len(rows), "rows")

for i, p in enumerate(images):
    if p.name in done:
        continue
    ns = scores.get(p.name, {"score": 0.0, "parts": ""})
    score = ns["score"]
    meta = manifest.get(p.name, {})
    row = {"file": meta.get("display_name", p.name), "name": p.name,
           "sha256": "", "size_mb": round(p.stat().st_size / 1024 / 1024, 3),
           "nsfw_score": score, "nsfw_parts": ns["parts"],
           "category": "other", "subcategory": "other", "confidence": 0.0,
           "caption": "", "needs_review": False,
           "media_id": meta.get("media_id", ""), "display_name": meta.get("display_name", p.name)}
    if score >= NSFW_HIGH:
        row.update(category="nsfw_explicit/other", subcategory="other",
                   confidence=round(score, 2), caption="[redacted]", needs_review=True)
    else:
        try:
            cat, sub, conf, cap = vlm_classify(p)
        except Exception as e:
            cat, sub, conf, cap = "other", "other", 0.0, f"vision failed: {e}"
        row.update(category=cat, subcategory=sub, confidence=round(conf, 2), caption=cap)
        if score >= NSFW_MID and not cat.startswith("nsfw_"):
            row["needs_review"] = True
        if cat.startswith("nsfw_explicit"):
            row["caption"] = "[redacted]"
            row["needs_review"] = True
    rows.append(row)
    if len(rows) % 50 == 0:
        pd.DataFrame(rows).to_csv(partial, index=False)
        print(f"  classified {len(rows)}/{len(images)} (checkpointed)")
pd.DataFrame(rows).to_csv("/content/plan.csv", index=False)
pd.DataFrame(rows).to_csv(partial, index=False)
json.dump(rows, open("/content/plan.json", "w"), indent=2, ensure_ascii=False)
print("DONE:", len(rows), "rows -> /content/plan.csv")
print(pd.DataFrame(rows)["category"].value_counts().to_string())
print("needs_review:", sum(1 for r in rows if r["needs_review"]))


In [ ]:
# 5 — Download the plan (do this BEFORE the wipe cell — the wipe deletes it from the runtime)
from google.colab import files
files.download("/content/plan.csv")
files.download("/content/plan.json")
print("Downloaded plan.csv + plan.json. Check they are on your computer, then run the wipe cell.")


In [ ]:
# 6 — WIPE: delete every thumbnail, the zip, scores and plans from this runtime. Run this last.
import shutil, pathlib
for target in ["/content/thumbs", "/content/thumbnails.zip", "/content/nudenet_scores.json",
               "/content/plan_partial.csv"]:
    p = pathlib.Path(target)
    if p.is_dir():
        shutil.rmtree(p, ignore_errors=True)
    elif p.exists():
        p.unlink()
# NOTE: plan.csv / plan.json are deleted here too — only run this AFTER downloading them.
for target in ["/content/plan.csv", "/content/plan.json"]:
    p = pathlib.Path(target)
    if p.exists():
        p.unlink()
print("remaining in /content:", [x.name for x in pathlib.Path("/content").iterdir()])
print("WIPED. If the list above is empty (or only sample_data), the thumbnails are gone from this runtime.")
print("Then close the tab — Colab recycles the runtime. In the app, tap Sort and pick the downloaded plan.csv.")
